# 7. Results visualization

Visualize the regenerated pipeline outputs. Refresh the packaged Tableau extract separately in Tableau Desktop after rerunning the pipeline.

Run the notebooks in numerical order, or run the entire workflow from the project root with `python -m src.run_pipeline`. See the README for setup and model assumptions.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the project folder or one of its subfolders.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import PROCESSED_DATA_DIR

forecast = pd.read_csv(PROCESSED_DATA_DIR / "forecast.csv", parse_dates=["Date"])
inventory = pd.read_csv(PROCESSED_DATA_DIR / "inventory_plan.csv")
production = pd.read_csv(PROCESSED_DATA_DIR / "production_plan.csv")
profit = pd.read_csv(PROCESSED_DATA_DIR / "profit_results.csv")

fig, ax = plt.subplots(figsize=(12, 5))
for product, group in forecast.groupby("Product"):
    ax.plot(group["Date"], group["Forecasted_Demand"], label=product)
ax.set(title="30-day demand forecast by product", xlabel="Date", ylabel="Forecasted units per day")
ax.legend(ncol=3, fontsize="small")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=inventory, x="Product", y="Safety_Stock", ax=axes[0])
sns.barplot(data=production, x="Product", y="Optimal_Production", ax=axes[1])
sns.barplot(data=profit, x="Product", y="Profit", ax=axes[2])
for ax, title in zip(axes, ["Safety stock", "Production plan", "Profit contribution"]):
    ax.set_title(title)
    ax.tick_params(axis="x", labelrotation=70)
plt.tight_layout()
plt.show()

profit.set_index("Product")[["Revenue", "Production_Cost", "Shortage_Total"]].plot(
    kind="bar", figsize=(12, 5), title="Revenue and costs by product"
)
plt.tight_layout()
plt.show()
